## 1. Session Overview

This notebook fine-tunes `meta-llama/Meta-Llama-3.1-8B-Instruct` using the MaxText framework on Kaggle TPU v5e.

- Objective: Run a minimal verification on TPU v5e, then proceed to fine-tuning.
- Evidence of Done: Successful `steps: 1` MaxText run and logs confirming TPU utilization.
- Artifacts: Config file, logs, and checkpoints saved to Kaggle outputs and/or GCS.

Preconditions:
- Kaggle accelerator set to TPU v5e.
- Internet enabled for cloning and dependency installs.
- Access to MaxText-compatible Llama 3.1 checkpoint via Kaggle Datasets.


## 2. Kaggle TPU v5e Environment Setup Plan

Steps in this session:
1. Verify TPU visibility and JAX version
2. Clone MaxText (main branch)
3. Install dependencies from `requirements.txt`
4. Prepare minimal `config.yaml` for verification run
5. Run a 1-step verification to confirm TPU v5e works

Notes:
- No substeps for now; each step maps to a single cell or small group of cells.
- We will capture logs and versions for reproducibility.


In [12]:
# 3. Verify TPU visibility and JAX environment
import os, sys, platform, subprocess, json

print("Python:", sys.version)
print("Platform:", platform.platform())

# Kaggle TPU env vars
for key in ["TPU_NAME", "TPU_WORKER_ID", "TPU_CHIPS_PER_PROCESS", "TPU_MULTISLICE_CTRL_ADDRESS"]:
    if key in os.environ:
        print(f"{key}:", os.environ[key])

try:
    import jax
    import jaxlib
    import jax.numpy as jnp
    print("jax:", jax.__version__)
    print("jaxlib:", jaxlib.__version__)
    devices = jax.devices()
    print("Devices:")
    for d in devices:
        print(" -", d)
    print("Device count:", len(devices))
    x = jnp.ones((8, 8))
    y = jnp.dot(x, x).block_until_ready()
    print("JAX test dot result shape:", y.shape)
except Exception as e:
    print("[ERROR] JAX/TPU verification failed:", e)
    raise


Python: 3.10.18 (main, Jul  1 2025, 05:26:40) [GCC 12.2.0]
Platform: Linux-6.1.42+-x86_64-with-glibc2.36
TPU_WORKER_ID: 0
jax: 0.4.34
jaxlib: 0.4.34
Devices:
 - TPU_0(process=0,(0,0,0,0))
 - TPU_1(process=0,(1,0,0,0))
 - TPU_2(process=0,(0,1,0,0))
 - TPU_3(process=0,(1,1,0,0))
 - TPU_4(process=0,(0,2,0,0))
 - TPU_5(process=0,(1,2,0,0))
 - TPU_6(process=0,(0,3,0,0))
 - TPU_7(process=0,(1,3,0,0))
Device count: 8
JAX test dot result shape: (8, 8)


## 4. Clone MaxText (main branch)

We will clone the official `google/maxtext` repository at the default `main` branch for the latest TPU v5e-compatible training scripts. Evidence of done: repository present in the working directory and HEAD commit printed.


In [13]:
%%bash
set -e

echo "Cloning google/maxtext (main)..."
if [ ! -d "maxtext" ]; then
  git clone --depth=1 https://github.com/google/maxtext.git
else
  echo "Repository 'maxtext' already exists; skipping clone."
fi

cd maxtext
echo "Repo HEAD:"
git log -1 --pretty=oneline || true

echo "Top-level files:"
ls -1 | sed -n '1,50p'


Cloning google/maxtext (main)...
Repository 'maxtext' already exists; skipping clone.
Repo HEAD:
a55e18af31a76179e589314878af0a5195e7d7bd Merge pull request #2278 from AI-Hypercomputer:collabs-examples-sft
Top-level files:
AUTHORS
CONTRIBUTING.md
LICENSE
PREFLIGHT.md
README.md
RESTRUCTURE.md
benchmarks
clean_py_env.Dockerfile
code_style.sh
docker_build_dependency_image.sh
docker_upload_runner.sh
docs
download_dataset.sh
end_to_end
gpu_multi_process_run.sh
maxtext_custom_wheels.Dockerfile
maxtext_db_dependencies.Dockerfile
maxtext_dependencies.Dockerfile
maxtext_gpu_dependencies.Dockerfile
maxtext_jax_ai_image.Dockerfile
maxtext_libtpu_path.Dockerfile
maxtext_runner.Dockerfile
multihost_job.py
multihost_runner.py
pedagogical_examples
preflight.sh
pylintrc
pyproject.toml
pytest.ini
requirements.txt
requirements_docs.txt
requirements_with_jax_ai_image.txt
requirements_with_jax_stable_stack_0_6_1_pipreqs.txt
rto_setup.sh
setup.sh
setup_gcsfuse.sh
setup_with_retries.sh
src
tests
unit_test_a

## Notes: Handling TensorFlow conflicts on TPU v5e

- Import order: Avoid importing TensorFlow before JAX; it can block TPU init.
- If TF causes conflicts but is not needed for JAX training, consider uninstalling `tensorflow` and using `tensorflow-cpu` instead.
- Keep JAX/jaxlib versions consistent with preinstalled TPU runtime.
- Evidence to capture on failure: exact import stack, package versions, and full error logs.


## 5. Install dependencies from requirements.txt

Install Python dependencies required by MaxText. Kaggle TPU v5e includes a modern JAX stack; if a conflict arises, we will prefer the preinstalled JAX. Evidence of done: successful pip install and import checks.


In [14]:
%%bash
set -e

echo "Updating apt and installing pkg-config..."
apt-get update && apt-get install -y pkg-config

echo "Upgrading pip..."
pip install --upgrade pip

echo "Installing MaxText requirements..."
# Now run the pip install command, which should find the newly installed pkg-config
pip install --no-input --no-cache-dir -r maxtext/requirements.txt

echo "Verifying JAX installation post-install..."
python - <<'PY'
import jax, jaxlib
print("jax:", jax.__version__)
print("jaxlib:", jaxlib.__version__)
print("JAX import successful after requirements install.")
PY

Updating apt and installing pkg-config...
Hit:1 http://deb.debian.org/debian bookworm InRelease
Hit:2 http://deb.debian.org/debian bookworm-updates InRelease
Hit:3 http://deb.debian.org/debian-security bookworm-security InRelease
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
pkg-config is already the newest version (1.8.1-1).
0 upgraded, 0 newly installed, 0 to remove and 103 not upgraded.
Upgrading pip...


Installing MaxText requirements...
     \ 538.6 kB 5.2 MB/s 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
     \ 2.8 MB 6.0 MB/s 0:00:000m
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     - 417.8 kB 14.2 MB/s 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     | 2.1 MB 11.7 MB/s 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml

Verifying JAX installation post-install...
jax: 0.4.34
jaxlib: 0.4.34
JAX import successful after requirements install.


## 6. Run 1-step MaxText verification

We will run a single training step using synthetic data. This should initialize JAX on TPU v5e and produce minimal logs without requiring a dataset.

Evidence of done:
- Process completes without error
- Logs show TPU devices used
- Step 1/1 completes


In [15]:
# Generate minimal config inline and save to /kaggle/working
from pathlib import Path

config_text = """
run_name: "verification_run_1step"
base_output_directory: "/kaggle/working/maxtext_runs"
dataset_type: "synthetic"
steps: 1
per_device_batch_size: 1
""".strip() + "\n"

out_path = Path("/kaggle/working/verification_minimal.yml")
out_path.write_text(config_text)
print("Wrote config to:", out_path)
print("\n--- Config ---\n" + out_path.read_text())


Wrote config to: /kaggle/working/verification_minimal.yml

--- Config ---
run_name: "verification_run_1step"
base_output_directory: "/kaggle/working/maxtext_runs"
dataset_type: "synthetic"
steps: 1
per_device_batch_size: 1



In [18]:
# Final verification run with the correct, verified path
import os, subprocess, sys

# The exact path you found using the 'find' command
train_script_path = "maxtext/src/MaxText/train.py"
config_path = "/kaggle/working/verification_minimal.yml"

print(f"Running script at verified path: {train_script_path}")

cmd = [sys.executable, train_script_path, config_path]
proc = subprocess.run(cmd, capture_output=True, text=True)

# Print both stdout and stderr for better debugging
print(proc.stdout)
if proc.stderr:
    print("--- stderr ---")
    print(proc.stderr)

if proc.returncode != 0:
    print(f"[ERROR] Verification run failed with return code {proc.returncode}")
else:
    print("[OK] Verification run completed.")

Running script at verified path: maxtext/src/MaxText/train.py

--- stderr ---
Traceback (most recent call last):
  File "/kaggle/working/maxtext/src/MaxText/train.py", line 32, in <module>
    import tensorflow as tf
  File "/usr/local/lib/python3.10/site-packages/tensorflow/__init__.py", line 37, in <module>
    from tensorflow.python.tools import module_util as _module_util
  File "/usr/local/lib/python3.10/site-packages/tensorflow/python/__init__.py", line 37, in <module>
    from tensorflow.python.eager import context
  File "/usr/local/lib/python3.10/site-packages/tensorflow/python/eager/context.py", line 29, in <module>
    from tensorflow.core.framework import function_pb2
  File "/usr/local/lib/python3.10/site-packages/tensorflow/core/framework/function_pb2.py", line 16, in <module>
    from tensorflow.core.framework import attr_value_pb2 as tensorflow_dot_core_dot_framework_dot_attr__value__pb2
  File "/usr/local/lib/python3.10/site-packages/tensorflow/core/framework/attr_valu

In [17]:
!find maxtext -name "train.py"

/usr/local/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


maxtext/src/MaxText/train.py
